In [3]:
import wikipediaapi
import json

def get_category_urls(category_obj, urls_set, limit=500, level=0, max_level=1):
    """Recursively crawls categories to find articles."""
    if level > max_level or len(urls_set) >= limit:
        return

    # members includes both Subcategories (Category:) and Articles
    for member in category_obj.categorymembers.values():
        if len(urls_set) >= limit:
            break

        if member.ns == wikipediaapi.Namespace.MAIN:
            # It's an article
            urls_set.add((member.title, member.fullurl))
        elif member.ns == wikipediaapi.Namespace.CATEGORY:
            # It's a subcategory, crawl it if we haven't reached max_level
            get_category_urls(member, urls_set, limit, level + 1, max_level)

def run_collection():
    wiki = wikipediaapi.Wikipedia(
        user_agent="WW2DataCollector/1.0 (contact@example.com)",
        language='en'
    )
    
    start_cat = wiki.page("Category:World War II")
    unique_articles = set() # Use a set to prevent duplicates across categories
    
    get_category_urls(start_cat, unique_articles, limit=500)
    
    # Format for JSON
    output_data = [{"title": t, "url": u} for t, u in unique_articles]
    
    with open('fixed_urls.json', 'w') as f:
        json.dump(output_data, f, indent=4)
    
    print(f"Successfully collected {len(output_data)} unique URLs.")

run_collection()

Successfully collected 500 unique URLs.


In [4]:
import json
import uuid
import wikipediaapi
from langchain_text_splitters import RecursiveCharacterTextSplitter

def process_and_chunk():
    with open('fixed_urls.json', 'r') as f:
        articles = json.load(f)

    # Tiktoken ensures the 200-400 range refers to actual model tokens
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name="gpt-4",
        chunk_size=400,
        chunk_overlap=50
    )

    wiki = wikipediaapi.Wikipedia(user_agent="WW2DataCollector/1.0", language='en')
    final_chunks = []

    for item in articles:
        page = wiki.page(item['title'])
        if not page.exists(): continue
        
        # We use .text which excludes sidebars/metadata by default in wikipediaapi
        raw_text = page.text.strip()
        
        # Split into chunks
        chunks = text_splitter.split_text(raw_text)

        for i, chunk_content in enumerate(chunks):
            final_chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "url": item['url'],
                "title": item['title'],
                "content": chunk_content,
                "metadata": {
                    "chunk_index": i,
                    "total_article_chunks": len(chunks)
                }
            })

    with open('ww2_chunks.json', 'w') as f:
        json.dump(final_chunks, f, indent=4)
    
    print(f"Processing complete. {len(final_chunks)} total chunks stored.")

process_and_chunk()

Processing complete. 7515 total chunks stored.
